# Pretrained ResNet Alice/Bob vs Fresh Attackers

This self-contained Colab notebook uses the real `torchvision` **ResNet-18 with ImageNet pretrained weights** as Alice and Bob for 64-bit neural cryptography. It does not initialize or train a ResNet from scratch. The pretrained backbone is retained, only its final residual block and a new 64-value output head are fine-tuned, and fresh attackers test the frozen pair.

Select a GPU runtime and run all cells. Nothing is saved to disk: progress, JSON logs, final results, and the attack matrix are printed into the notebook output. This is a research experiment, not production encryption.

## 1. Imports and settings

The notebook detects an A100 or L4 and selects a large safe batch automatically. **A100 is recommended** for the fastest run; L4 is also fully supported. Reduce the step counts only for a quick wiring test.

In [ ]:
import json, random
from contextlib import nullcontext
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
BITS = 32  # Message, key, and ciphertext width. Must be a multiple of 4.
PAIR_STEPS = 5_000
ATTACK_STEPS = 2_000
EVE_UPDATES = 3
EVAL_SIZE = 16_384
ELM_SAMPLES = 16_384
ELM_HIDDEN = 512
LOG_EVERY = 500
LR_AB, LR_EVE, PRIVACY_WEIGHT = 1e-3, 1e-3, 8.0
ATTACKERS = ["CNN", "MLP", "BiLSTM", "Transformer", "ELM"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
if "A100" in GPU_NAME:
    BATCH_SIZE = 2048
elif "L4" in GPU_NAME:
    BATCH_SIZE = 1024
elif device.type == "cuda":
    BATCH_SIZE = 128
else:
    BATCH_SIZE = 8
USE_AMP = device.type == "cuda"
AMP_DTYPE = torch.bfloat16 if ("A100" in GPU_NAME or "L4" in GPU_NAME) else torch.float16
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
if USE_AMP:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
try: scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
except AttributeError: scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)
print(f"GPU: {GPU_NAME}")
print(f"Batch size: {BATCH_SIZE:,}; AMP dtype: {AMP_DTYPE if USE_AMP else 'disabled'}; TF32: {USE_AMP}")
print(f"Pair steps={PAIR_STEPS:,}; attack steps={ATTACK_STEPS:,}")

## 2. Data and metrics

Readable progress is printed normally. Lines beginning with `LOG_JSON` and `RESULT_JSON` are stable machine-readable records preserved in saved notebook output.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def random_batch(size):
    message = 2 * torch.randint(0, 2, (size, BITS), device=device).float() - 1
    key = 2 * torch.randint(0, 2, (size, BITS), device=device).float() - 1
    return message, key

def fixed_batch(size, seed):
    g = torch.Generator().manual_seed(seed)
    message = 2 * torch.randint(0, 2, (size, BITS), generator=g).float() - 1
    key = 2 * torch.randint(0, 2, (size, BITS), generator=g).float() - 1
    return message.to(device), key.to(device)

def error(output, target):
    return torch.abs((output + 1) / 2 - (target + 1) / 2).sum(1).mean()

def metrics(output, target):
    guess = torch.where(output >= 0, 1.0, -1.0)
    return {"bit_accuracy": (guess == target).float().mean().item(),
            "full_message_accuracy": (guess == target).all(1).float().mean().item(),
            "hamming_error": (guess != target).float().sum(1).mean().item()}

def log(event, **values):
    print("LOG_JSON " + json.dumps({"event": event, **values}, sort_keys=True), flush=True)

def amp(): return torch.autocast("cuda", dtype=AMP_DTYPE) if USE_AMP else nullcontext()
def update(loss, optimizer):
    optimizer.zero_grad(set_to_none=True); scaler.scale(loss).backward()
    scaler.step(optimizer); scaler.update()

set_seed(SEED)

## 3. Models

Alice receives 128 values (64-bit message + 64-bit key), and Bob receives 128 values (64-value ciphertext + 64-bit key). Each input becomes an 8×16 three-channel image and is resized to ResNet's standard 224×224 input. The actual ImageNet-pretrained stem through `layer3` stays frozen; only `layer4` and the new 64-value output head are fine-tuned. This is transfer learning, not training ResNet from scratch.

In [ ]:
class ResNetBits(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = ResNet18_Weights.DEFAULT
        self.net = resnet18(weights=self.weights)  # official pretrained weights
        self.net.fc = nn.Linear(self.net.fc.in_features, BITS)
        for p in self.net.parameters(): p.requires_grad_(False)
        for p in self.net.layer4.parameters(): p.requires_grad_(True)
        for p in self.net.fc.parameters(): p.requires_grad_(True)
        transform = self.weights.transforms()
        self.register_buffer("mean", torch.tensor(transform.mean).view(1,3,1,1))
        self.register_buffer("std", torch.tensor(transform.std).view(1,3,1,1))
    def train(self, mode=True):
        super().train(mode)
        if mode:
            self.net.eval(); self.net.layer4.train(); self.net.fc.train()
        return self
    def forward(self, x):
        assert BITS % 4 == 0, "BITS must be divisible by 4 to form the ResNet input grid"
        assert x.shape[1] == 2 * BITS, f"Expected {2 * BITS} values, got {x.shape[1]}"
        # Keep the leading dimension explicit: the old fixed 8x16 reshape
        # silently merged two 32-bit examples into one batch element.
        grid = ((x + 1) / 2).reshape(x.shape[0], 1, 8, BITS // 4)
        image = torch.cat([grid, grid.flip(2), grid.flip(3)], 1)
        image = F.interpolate(image, (224,224), mode="nearest")
        image = image.contiguous(memory_format=torch.channels_last)
        return torch.tanh(self.net((image - self.mean) / self.std))

class CNN(nn.Module):
    def __init__(self):
        super().__init__(); self.net = nn.Sequential(
            nn.Conv1d(1,32,5,padding=2),nn.ReLU(),nn.Conv1d(32,64,3,padding=1),
            nn.ReLU(),nn.Flatten(),nn.Linear(64*BITS,BITS),nn.Tanh())
    def forward(self,x): return self.net(x.unsqueeze(1))

class MLP(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Linear(BITS,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),nn.Linear(256,BITS),nn.Tanh())
    def forward(self,x): return self.net(x)

class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__(); self.rnn=nn.LSTM(1,32,2,batch_first=True,bidirectional=True); self.out=nn.Linear(64,1)
    def forward(self,x): return torch.tanh(self.out(self.rnn(x.unsqueeze(-1))[0]).squeeze(-1))

class Transformer(nn.Module):
    def __init__(self):
        super().__init__(); self.project=nn.Linear(1,32); self.position=nn.Parameter(torch.zeros(1,BITS,32))
        layer=nn.TransformerEncoderLayer(32,4,64,dropout=0,batch_first=True,activation="gelu")
        self.encoder=nn.TransformerEncoder(layer,2); self.out=nn.Linear(32,1)
    def forward(self,x):
        return torch.tanh(self.out(self.encoder(self.project(x.unsqueeze(-1))+self.position)).squeeze(-1))

def make_eve(name): return {"CNN":CNN,"MLP":MLP,"BiLSTM":BiLSTM,"Transformer":Transformer}[name]().to(device)

def print_model_size(name, model):
    tensors = list(model.parameters()) + list(model.buffers())
    parameter_count = sum(p.numel() for p in model.parameters())
    size_bytes = sum(t.numel() * t.element_size() for t in tensors)
    print(f"{name} size: {parameter_count:,} parameters; {size_bytes / 2**20:.2f} MiB (parameters + buffers)")

def print_elm_size(attacker):
    tensors = [attacker.w, attacker.b] + ([] if attacker.beta is None else [attacker.beta])
    value_count = sum(t.numel() for t in tensors)
    size_bytes = sum(t.numel() * t.element_size() for t in tensors)
    print(f"ELM size: {value_count:,} stored values; {size_bytes / 2**20:.2f} MiB (including fitted output weights)")

class ELM:
    def __init__(self, seed):
        g=torch.Generator().manual_seed(seed)
        self.w=(torch.randn(BITS,ELM_HIDDEN,generator=g)/BITS**.5).to(device)
        self.b=torch.randn(ELM_HIDDEN,generator=g).to(device); self.beta=None
    def hidden(self,x): return torch.tanh(x@self.w+self.b)
    def fit(self,x,y):
        h=self.hidden(x).float(); gram=h.T@h; gram.diagonal().add_(1e-3)
        self.beta=torch.linalg.solve(gram,h.T@y.float())
    def __call__(self,x): return torch.tanh(self.hidden(x).float()@self.beta)

alice = ResNetBits().to(device, memory_format=torch.channels_last)
bob = ResNetBits().to(device, memory_format=torch.channels_last)
total = sum(p.numel() for p in alice.parameters())
trainable = sum(p.numel() for p in alice.parameters() if p.requires_grad)
print(f"Loaded genuine pretrained weights: {alice.weights}")
print(f"Parameters per model: {total:,} total; {trainable:,} fine-tuned; {total-trainable:,} frozen")

## 4. Train Alice and Bob against a CNN Eve

In [ ]:
eve=CNN().to(device); print_model_size("Training CNN Eve", eve)
ab_opt=torch.optim.AdamW([p for model in (alice,bob) for p in model.parameters() if p.requires_grad],lr=LR_AB)
eve_opt=torch.optim.Adam(eve.parameters(),lr=LR_EVE)
alice.train(); bob.train(); eve.train()

for step in range(1,PAIR_STEPS+1):
    for _ in range(EVE_UPDATES):
        message,key=random_batch(BATCH_SIZE)
        with torch.no_grad(),amp(): ciphertext=alice(torch.cat([message,key],1))
        with amp(): eve_loss=error(eve(ciphertext),message)
        update(eve_loss,eve_opt)
    message,key=random_batch(BATCH_SIZE); eve.requires_grad_(False)
    with amp():
        ciphertext=alice(torch.cat([message,key],1)); bob_out=bob(torch.cat([ciphertext,key],1)); eve_out=eve(ciphertext)
        bob_loss=error(bob_out,message); eve_error=error(eve_out,message)
        total=bob_loss+PRIVACY_WEIGHT*((BITS/2-eve_error)**2/(BITS/2)**2)
    update(total,ab_opt); eve.requires_grad_(True)
    if step==1 or step%LOG_EVERY==0 or step==PAIR_STEPS:
        bm,em=metrics(bob_out,message),metrics(eve_out,message)
        print(f"Pair {step:>5,}/{PAIR_STEPS:,} | Bob {bm['bit_accuracy']:.2%} | Eve {em['bit_accuracy']:.2%} | Bob L1 {bob_loss.item():.2f}")
        log("pair_train",step=step,bob_loss=bob_loss.item(),eve_loss=eve_error.item(),bob_accuracy=bm["bit_accuracy"],eve_accuracy=em["bit_accuracy"])

alice.eval(); bob.eval(); alice.requires_grad_(False); bob.requires_grad_(False)
print("Alice and Bob frozen; starting fresh attacks.")

## 5. Train fresh attackers and evaluate them on the same held-out set

In [ ]:
test_message,test_key=fixed_batch(EVAL_SIZE,SEED+10_000)
test_cipher_parts, test_bob_parts = [], []
with torch.inference_mode():
    for start in range(0,EVAL_SIZE,BATCH_SIZE):
        message_part=test_message[start:start+BATCH_SIZE]
        key_part=test_key[start:start+BATCH_SIZE]
        with amp():
            cipher_part=alice(torch.cat([message_part,key_part],1))
            bob_part=bob(torch.cat([cipher_part,key_part],1))
        test_cipher_parts.append(cipher_part.float())
        test_bob_parts.append(bob_part.float())
test_cipher=torch.cat(test_cipher_parts); test_bob=torch.cat(test_bob_parts)
bob_score=metrics(test_bob,test_message); results=[]

for index,name in enumerate(ATTACKERS):
    set_seed(SEED+100+index); print(f"\nTraining {name} attacker")
    if name=="ELM":
        train_message,train_key=fixed_batch(ELM_SAMPLES,SEED+20_000); chunks=[]
        with torch.inference_mode():
            for start in range(0,ELM_SAMPLES,BATCH_SIZE):
                pair=torch.cat([train_message[start:start+BATCH_SIZE],train_key[start:start+BATCH_SIZE]],1)
                with amp(): chunks.append(alice(pair).float())
        attacker=ELM(SEED+100+index); attacker.fit(torch.cat(chunks),train_message)
        print(f"ELM fitted on {ELM_SAMPLES:,} examples"); print_elm_size(attacker); log("attacker_train",attacker=name,samples=ELM_SAMPLES)
    else:
        attacker=make_eve(name).train(); print_model_size(f"{name} attacker", attacker); optimizer=torch.optim.Adam(attacker.parameters(),lr=LR_EVE)
        for step in range(1,ATTACK_STEPS+1):
            message,key=random_batch(BATCH_SIZE)
            with torch.no_grad(),amp(): ciphertext=alice(torch.cat([message,key],1))
            with amp(): output=attacker(ciphertext); loss=error(output,message)
            update(loss,optimizer)
            if step==1 or step%LOG_EVERY==0 or step==ATTACK_STEPS:
                score=metrics(output,message); print(f"  {step:>5,}/{ATTACK_STEPS:,} | accuracy {score['bit_accuracy']:.2%} | L1 {loss.item():.2f}")
                log("attacker_train",attacker=name,step=step,loss=loss.item(),accuracy=score["bit_accuracy"])
        attacker.eval()
    with torch.inference_mode(): attack_score=metrics(attacker(test_cipher.float()).float(),test_message)
    row={"alice_bob":"Pretrained ResNet-18","training_eve":"CNN","attacker":name,
         "bob_bit_accuracy":bob_score["bit_accuracy"],"bob_full_message_accuracy":bob_score["full_message_accuracy"],
         "attacker_bit_accuracy":attack_score["bit_accuracy"],"attacker_full_message_accuracy":attack_score["full_message_accuracy"],
         "attacker_hamming_error":attack_score["hamming_error"]}
    results.append(row); print("RESULT_JSON "+json.dumps(row,sort_keys=True),flush=True)
    del attacker
    if device.type=="cuda": torch.cuda.empty_cache()

## 6. Print and plot the final attack matrix

An attacker accuracy near 50% is random guessing. Higher accuracy means that attacker recovered plaintext information. Save the Colab notebook itself to preserve these printed outputs.

In [ ]:
results_df=pd.DataFrame(results)
matrix=results_df.pivot(index="alice_bob",columns="attacker",values="attacker_bit_accuracy").reindex(columns=ATTACKERS)
print("FINAL RESULTS")
print(results_df.to_string(index=False,float_format=lambda x:f"{x:.4f}"))
print("\nATTACK MATRIX — held-out bit accuracy (50% = random)")
print(matrix.to_string(float_format=lambda x:f"{x:.2%}"))
display(matrix.style.format("{:.2%}").set_caption("Fresh attackers vs pretrained ResNet-18 Alice/Bob"))

fig,ax=plt.subplots(figsize=(9,2.7)); image=ax.imshow(matrix.to_numpy(),vmin=.5,vmax=1,cmap="magma",aspect="auto")
ax.set_xticks(range(len(matrix.columns)),matrix.columns); ax.set_yticks([0],matrix.index)
ax.set_xlabel("Fresh post-hoc attacker"); ax.set_title("Attack matrix: held-out bit accuracy")
for column,value in enumerate(matrix.iloc[0]): ax.text(column,0,f"{value:.1%}",ha="center",va="center",color="white" if value<.7 else "black")
fig.colorbar(image,ax=ax,label="bit accuracy"); plt.show()
print(f"\nBob held-out bit accuracy: {bob_score['bit_accuracy']:.2%}")
print("No files were written. These notebook outputs are the experiment record.")